# Monitoring a credit scorecard through a downturn

A model is trained on a stable retail-credit portfolio and deployed. Months later the macro environment turns: FICO scores fall, debt-to-income rises, revolving utilisation climbs.

Nothing about the model has changed. Nothing about the *relationship* between the drivers and default has changed either — this is pure covariate shift. The question every monitoring pack has to answer is:

1. Has the population moved? (PSI)
2. Which inputs moved it? (CSI)
3. Does the model still discriminate? (AUC)
4. Do its probabilities still mean what they claim? (calibration)

Those last two fail independently, and this notebook shows a case where one holds and the other does not.

In [1]:
import pandas as pd

from driftkit import (
    WOEEncoder,
    calibration_report,
    csi,
    fit_bins,
    make_credit_data,
    monitor,
    psi,
)

pd.set_option("display.precision", 4)

FEATURES = [
    "fico_score",
    "debt_to_income",
    "revol_util",
    "delinq_2yrs",
    "text_sentiment",
    "age",
]

## 1. Two populations

`make_credit_data` is a synthetic generator: default propensity is a logistic function of interpretable drivers. Because the ground truth is written down in code, we can shift a driver by a known amount and check the metrics notice.

`age` is generated identically in both scenarios. It is the control — any metric that flags it is producing noise.

In [2]:
reference = make_credit_data(40_000, seed=0)  # training population
current = make_credit_data(40_000, drift=True, seed=1)  # this quarter

pd.DataFrame(
    {
        "reference": reference[[*FEATURES, "default_event"]].mean(),
        "current": current[[*FEATURES, "default_event"]].mean(),
    }
).assign(change=lambda d: d["current"] - d["reference"])

,reference,current,change
fico_score,699.8169,659.5111,-40.3058
debt_to_income,0.2283,0.3856,0.1573
revol_util,0.4001,0.5996,0.1995
delinq_2yrs,0.2213,0.4727,0.2515
text_sentiment,0.1985,-0.0917,-0.2901
age,48.4401,48.3704,-0.0697
default_event,0.1012,0.4038,0.3026


The realised default rate has quadrupled, from 10.1% to 40.4%. A table of means tells you *that*, but not how much of it is population mix versus model failure. That is what the rest of this notebook separates.

## 2. Which features moved

`monitor` runs PSI across every shared column. Bins are learned on the reference frame and applied unchanged to the current one.

In [3]:
report = monitor(reference[FEATURES], current[FEATURES])
report.summary

,feature,psi,interpretation
0,debt_to_income,1.0821,significant shift
1,revol_util,0.9065,significant shift
2,fico_score,0.5144,significant shift
3,text_sentiment,0.4454,significant shift
4,delinq_2yrs,0.1329,moderate shift
5,age,0.0003,stable


`age` sits at the bottom, near zero, exactly as it should — that is the sanity check that the ranking above is signal rather than binning artefact.

A headline PSI is only actionable if you can see which part of the distribution moved:

In [4]:
report.results["fico_score"].top_bins.head(5)

,bin,expected_count,actual_count,expected_pct,actual_pct,contribution
0,"[-inf, 635.646)",4000.0,13769.0,0.1,0.3442,0.3018
9,"[763.888, inf)",4000.0,1613.0,0.1,0.0403,0.0542
8,"[741.75, 763.888)",4000.0,1755.0,0.1,0.0439,0.0462
7,"[726.047, 741.75)",4000.0,1901.0,0.1,0.0475,0.0390
6,"[712.675, 726.047)",4000.0,2136.0,0.1,0.0534,0.0292


## 3. Why re-binning destroys the metric

The most common PSI implementation applies `qcut` to each sample separately. Quantile bins are equal-frequency *by construction*, so both histograms come out uniform and the statistic collapses — regardless of how far the population actually moved.

In [5]:
from driftkit.drift import psi_from_counts

correct = psi(reference["fico_score"], current["fico_score"]).value

# The bug: bins refitted on each sample independently.
naive = psi_from_counts(
    fit_bins(reference["fico_score"], 10).counts(reference["fico_score"]),
    fit_bins(current["fico_score"], 10).counts(current["fico_score"]),
).value

print(f"frozen reference bins : {correct:.4f}")
print(f"refit per sample      : {naive:.4f}")

frozen reference bins : 0.5144
refit per sample      : 0.0000


Not merely understated — exactly zero. A monitoring pack built this way reports "stable" forever.

## 4. Fit a scorecard

WOE-encode the reference data and fit a logistic regression. WOE linearises each feature against the log-odds of default, which is what keeps a scorecard monotone and convertible to points.

Note that the encoder is fitted on the reference data **only** and then applied to the current period. Refitting it on current data would silently absorb the drift we are trying to measure.

In [6]:
encoder = WOEEncoder(n_bins=10).fit(reference[FEATURES], reference["default_event"])

pd.Series(encoder.information_values_, name="IV").sort_values(ascending=False).to_frame()

,IV
fico_score,0.4376
text_sentiment,0.2170
debt_to_income,0.1623
revol_util,0.1167
delinq_2yrs,0.1160
age,0.0031


IV ranks predictive strength. `age` is near zero — it carries no signal, which matches how the data was generated. A very high IV would be a leakage warning rather than good news.

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

X_reference = encoder.transform(reference[FEATURES])
X_current = encoder.transform(current[FEATURES])

model = LogisticRegression(max_iter=1000).fit(X_reference, reference["default_event"])

p_reference = model.predict_proba(X_reference)[:, 1]
p_current = model.predict_proba(X_current)[:, 1]

print(f"AUC reference : {roc_auc_score(reference['default_event'], p_reference):.4f}")
print(f"AUC current   : {roc_auc_score(current['default_event'], p_current):.4f}")

AUC reference : 0.7769
AUC current   : 0.7789


**AUC barely moves.** Rank-ordering survives the downturn, because the relationship between drivers and default genuinely did not change.

If AUC were the only thing on the dashboard, this model would be declared healthy.

## 5. The score distribution moved

PSI on the model output is the standard portfolio-level tripwire.

In [8]:
score_drift = psi(p_reference, p_current)
print(score_drift)

DriftResult(value=2.0399, interpretation='significant shift', n_expected=40000, n_actual=40000)


### Attributing the move to a feature

PSI on the output says the scored population moved. CSI on an input says which one moved it — and with scorecard points attached, how many points that is worth.

In [9]:
spec = fit_bins(reference["fico_score"], n_bins=10)

# Points per bin: the scorecard's own coefficient times each bin's WOE.
coefficient = model.coef_[0][FEATURES.index("fico_score")]
points = encoder.woe_maps_["fico_score"] * coefficient

fico_csi = csi(reference["fico_score"], current["fico_score"], points=points, bins=spec)

print(f"CSI            : {fico_csi.value:.4f}")
print(f"log-odds shift : {fico_csi.table['score_impact'].sum():+.4f}")
fico_csi.table[["bin", "expected_pct", "actual_pct", "points", "score_impact"]]

CSI            : 0.5144
log-odds shift : +0.5109


,bin,expected_pct,actual_pct,points,score_impact
0,"[-inf, 635.646)",9.9999e-02,3.4419e-01,1.0918,0.2666
1,"[635.646, 657.606)",9.9999e-02,1.4277e-01,0.5797,0.0248
2,"[657.606, 673.699)",9.9999e-02,1.0587e-01,0.2697,0.0016
3,"[673.699, 687.117)",9.9999e-02,8.6301e-02,0.2452,-0.0034
4,"[687.117, 699.887)",9.9999e-02,7.4152e-02,-0.0146,0.0004
5,"[699.887, 712.675)",9.9999e-02,6.1554e-02,-0.4532,0.0174
6,"[712.675, 726.047)",9.9999e-02,5.3405e-02,-0.4872,0.0227
7,"[726.047, 741.75)",9.9999e-02,4.7531e-02,-0.6478,0.0340
8,"[741.75, 763.888)",9.9999e-02,4.3881e-02,-1.0184,0.0572
9,"[763.888, inf)",9.9999e-02,4.0332e-02,-1.5013,0.0896


## 6. Calibration — where it actually breaks

AUC held. Now check whether the probabilities still mean what they say.

In [10]:
before = calibration_report(reference["default_event"], p_reference)
after = calibration_report(current["default_event"], p_current)

pd.DataFrame(
    {
        "reference": [
            before.brier,
            before.ece,
            before.bias,
            before.predicted_rate,
            before.observed_rate,
        ],
        "current": [
            after.brier,
            after.ece,
            after.bias,
            after.predicted_rate,
            after.observed_rate,
        ],
    },
    index=["brier", "ece", "bias", "predicted_rate", "observed_rate"],
)

,reference,current
brier,8.0017e-02,0.1913
ece,3.1003e-03,0.0653
bias,-3.4580e-05,-0.0653
predicted_rate,1.0119e-01,0.3386
observed_rate,1.0122e-01,0.4038


The reliability table shows where the gap opens up across the probability range:

In [11]:
after.table.dropna()

,bin,count,predicted_rate,observed_rate,gap
0,"[0.00, 0.10)",5192.0,0.0616,0.0749,-0.0133
1,"[0.10, 0.20)",7520.0,0.1492,0.1866,-0.0374
2,"[0.20, 0.30)",6680.0,0.2495,0.3121,-0.0626
3,"[0.30, 0.40)",6130.0,0.3482,0.4289,-0.0807
4,"[0.40, 0.50)",4768.0,0.4476,0.5434,-0.0958
5,"[0.50, 0.60)",4212.0,0.5468,0.6486,-0.1018
6,"[0.60, 0.70)",2905.0,0.6450,0.7401,-0.0951
7,"[0.70, 0.80)",1871.0,0.7402,0.8172,-0.0770
8,"[0.80, 0.90)",722.0,0.8386,0.8947,-0.0561


### Reading the decomposition

Murphy's decomposition splits the Brier score into `reliability - resolution + uncertainty`. It matters because the Brier score moves when the *base rate* moves even if the model is untouched — `uncertainty` is a property of the sample, not the model.

In [12]:
pd.DataFrame(
    {
        "reference": [before.reliability, before.resolution, before.uncertainty, before.brier],
        "current": [after.reliability, after.resolution, after.uncertainty, after.brier],
    },
    index=[
        "reliability (lower better)",
        "resolution (higher better)",
        "uncertainty (sample)",
        "brier",
    ],
)

,reference,current
reliability (lower better),5.9042e-05,0.0051
resolution (higher better),1.0347e-02,0.0536
uncertainty (sample),9.0978e-02,0.2408
brier,8.0017e-02,0.1913


## What the monitoring pack should say

| Question | Metric | Verdict |
| --- | --- | --- |
| Did the population move? | PSI on inputs and score | Yes — several features well past 0.25 |
| Which inputs drove it? | CSI | DTI, utilisation, FICO, sentiment |
| Does it still rank-order? | AUC | Yes, essentially unchanged |
| Are the probabilities still right? | ECE / bias | No — the model under-predicts the new default rate |

The action this implies is **recalibration, not retraining**. The learned relationship is intact; only the intercept is stale. Retraining would be the expensive answer to a question nobody asked — and a dashboard carrying AUC alone would have recommended doing nothing at all.

One caveat worth keeping in view: this notebook has labels for the current period. In production you usually do not, because defaults take months to mature. Until they do, PSI and CSI on the inputs are the only signals available — which is precisely why getting them arithmetically right matters.